# Phase 6.0 Final Candidate Confirmation

This notebook starts the final confirmation stage of the project.

Phase 5 established:

- a locked structured benchmark:
  - `structured_winner`
- a strong linear standalone reference:
  - `ridge_direct_full`
- a pruned standalone nonlinear reference:
  - `histgb_pruned_v1_no_anchor_no_regime`
- promising hybrid candidates from earlier exploratory work

The main Phase 6 question is:

**Does the hybrid still win after rebuilding it with the updated pruned HistGB nonlinear branch?**

This notebook is a confirmation notebook, not a broad search notebook.

## Phase 6 scope
This notebook will:

1. rebuild the finalist candidates in a fresh, self-contained setup
2. compare the updated finalists across both validation protocols
3. decide which candidates advance to final training-policy selection and tuning

## Locked candidate set entering Phase 6

Benchmark:
- `structured_winner`

Standalone references:
- `ridge_direct_full`
- `histgb_pruned_v1_no_anchor_no_regime`

Provisional hybrid candidates:
- `hybrid_center_structured_else_histgb_pruned`
- `hybrid_slice_no_hard_case_override_pruned`

Old full-HistGB-based hybrids may be compared briefly for backward reference, but are not the main focus.

## Decision rules
- Primary ranking metric: mean RMSE on `primary_realistic`
- Must also show acceptable confirmation on `stress_test`
- Prefer lower `max_two_protocols` when primary scores are close
- Prefer simpler models when performance is nearly tied
- Prefer more stable slice behavior when still tied


## Candidate registry convention

Each candidate recorded in this notebook will include:

- its model family
- target mode
- routing style
- nonlinear branch used
- what changed relative to Phase 5
- current role/status

This is important because Phase 5.1 changed the best standalone nonlinear branch from the full HistGB residual model to the pruned HistGB residual model.


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_root_files = (candidate / "train.csv").exists() and (candidate / "test.csv").exists()
        has_data_files = (candidate / "data" / "train.csv").exists() and (candidate / "data" / "test.csv").exists()
        if has_root_files or has_data_files:
            return candidate
    raise FileNotFoundError("Could not locate project root.")


def resolve_data_path(project_root: Path, filename: str) -> Path:
    for path in [project_root / filename, project_root / "data" / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename} in project root or data/")


PROJECT_ROOT = find_project_root()

train = pd.read_csv(resolve_data_path(PROJECT_ROOT, "train.csv"), parse_dates=["date"])
test = pd.read_csv(resolve_data_path(PROJECT_ROOT, "test.csv"), parse_dates=["date"])

for df in (train, test):
    if "tau" not in df.columns:
        df["tau"] = df["maturity_days"] / 365.0

train_date_summary = (
    train.groupby("date")
    .agg(
        total_rows=("row_id", "size"),
        observed_rows=("iv_observed", lambda s: s.notna().sum()),
        missing_rows=("iv_observed", lambda s: s.isna().sum()),
        observed_ratio=("iv_observed", lambda s: s.notna().mean()),
    )
    .sort_index()
)

train_dates = train_date_summary.index.to_list()

N_FOLDS = 4
VAL_DATES_PER_FOLD = 5
TOTAL_VAL_DATES = N_FOLDS * VAL_DATES_PER_FOLD
val_start_idx = len(train_dates) - TOTAL_VAL_DATES

fold_plan = pd.DataFrame(
    [
        {
            "fold": fold_id + 1,
            "train_start": train_dates[0],
            "train_end": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD - 1],
            "n_train_dates": val_start_idx + fold_id * VAL_DATES_PER_FOLD,
            "val_start": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD],
            "val_end": train_dates[val_start_idx + (fold_id + 1) * VAL_DATES_PER_FOLD - 1],
            "n_val_dates": VAL_DATES_PER_FOLD,
        }
        for fold_id in range(N_FOLDS)
    ]
)

print("Bootstrap check")
print("Project root:", PROJECT_ROOT)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(fold_plan)


Bootstrap check
Project root: /Users/sauravkrishna/Documents/projects/NQFO-Impilied-volatility-surface
Train shape: (11640, 10)
Test shape: (3960, 10)


,fold,train_start,train_end,n_train_dates,val_start,val_end,n_val_dates
0,1,2025-01-02,2025-04-18,77,2025-04-21,2025-04-25,5
1,2,2025-01-02,2025-04-25,82,2025-04-28,2025-05-02,5
2,3,2025-01-02,2025-05-02,87,2025-05-05,2025-05-09,5
3,4,2025-01-02,2025-05-09,92,2025-05-12,2025-05-16,5


In [3]:
phase6_candidate_registry = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "family": "structured",
            "target_mode": "direct_iv",
            "routing_style": "single_model",
            "nonlinear_branch": "(none)",
            "changed_from_phase5": "No change.",
            "status_entering_phase6": "benchmark",
            "notes": "Locked structured benchmark from Phase 4/5.",
        },
        {
            "model": "ridge_direct_full",
            "family": "linear_ml",
            "target_mode": "direct_iv",
            "routing_style": "single_model",
            "nonlinear_branch": "(none)",
            "changed_from_phase5": "No change.",
            "status_entering_phase6": "standalone_reference",
            "notes": "Best linear standalone reference and conservative specialist.",
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "family": "tree_ml",
            "target_mode": "residual",
            "routing_style": "single_model",
            "nonlinear_branch": "pruned_histgb_residual",
            "changed_from_phase5": "Replaces full HistGB residual as the main standalone nonlinear reference after 05_1 pruning.",
            "status_entering_phase6": "standalone_reference",
            "notes": "Best standalone nonlinear reference entering Phase 6.",
        },
        {
            "model": "hybrid_center_structured_else_histgb_pruned",
            "family": "hybrid",
            "target_mode": "mixed",
            "routing_style": "center_vs_wing",
            "nonlinear_branch": "histgb_pruned_v1_no_anchor_no_regime",
            "changed_from_phase5": "Updated from full HistGB branch to pruned HistGB branch.",
            "status_entering_phase6": "provisional_hybrid",
            "notes": "Phase 6 confirmation candidate; structured in center, pruned HistGB elsewhere.",
        },
        {
            "model": "hybrid_slice_no_hard_case_override_pruned",
            "family": "hybrid",
            "target_mode": "mixed",
            "routing_style": "slice_gated",
            "nonlinear_branch": "histgb_pruned_v1_no_anchor_no_regime",
            "changed_from_phase5": "Updated from full HistGB branch to pruned HistGB branch.",
            "status_entering_phase6": "provisional_hybrid",
            "notes": "Phase 6 confirmation candidate; default pruned HistGB, structured on center/quadratic, Ridge on tv_maturity_only.",
        },
        {
            "model": "hybrid_center_structured_else_histgb",
            "family": "hybrid",
            "target_mode": "mixed",
            "routing_style": "center_vs_wing",
            "nonlinear_branch": "histgb_residual_full",
            "changed_from_phase5": "No change; retained only for brief backward comparison.",
            "status_entering_phase6": "backward_reference",
            "notes": "Old full-HistGB-based hybrid for one-time comparison only.",
        },
        {
            "model": "hybrid_slice_no_hard_case_override",
            "family": "hybrid",
            "target_mode": "mixed",
            "routing_style": "slice_gated",
            "nonlinear_branch": "histgb_residual_full",
            "changed_from_phase5": "No change; retained only for brief backward comparison.",
            "status_entering_phase6": "backward_reference",
            "notes": "Old provisional hybrid winner from 05_0; needs re-confirmation with pruned branch.",
        },
    ]
)

print("Phase 6 candidate registry")
display(phase6_candidate_registry)


Phase 6 candidate registry


,model,family,target_mode,routing_style,nonlinear_branch,changed_from_phase5,status_entering_phase6,notes
0,structured_winner,structured,direct_iv,single_model,(none),No change.,benchmark,Locked structured benchmark from Phase 4/5.
1,ridge_direct_full,linear_ml,direct_iv,single_model,(none),No change.,standalone_reference,Best linear standalone reference and conservat...
2,histgb_pruned_v1_no_anchor_no_regime,tree_ml,residual,single_model,pruned_histgb_residual,Replaces full HistGB residual as the main stan...,standalone_reference,Best standalone nonlinear reference entering P...
3,hybrid_center_structured_else_histgb_pruned,hybrid,mixed,center_vs_wing,histgb_pruned_v1_no_anchor_no_regime,Updated from full HistGB branch to pruned Hist...,provisional_hybrid,Phase 6 confirmation candidate; structured in ...
4,hybrid_slice_no_hard_case_override_pruned,hybrid,mixed,slice_gated,histgb_pruned_v1_no_anchor_no_regime,Updated from full HistGB branch to pruned Hist...,provisional_hybrid,Phase 6 confirmation candidate; default pruned...
5,hybrid_center_structured_else_histgb,hybrid,mixed,center_vs_wing,histgb_residual_full,No change; retained only for brief backward co...,backward_reference,Old full-HistGB-based hybrid for one-time comp...
6,hybrid_slice_no_hard_case_override,hybrid,mixed,slice_gated,histgb_residual_full,No change; retained only for brief backward co...,backward_reference,Old provisional hybrid winner from 05_0; needs...


In [4]:
BUCKET_COLS = ["maturity_label", "option_type"]
NODE_COLS = ["maturity_label", "moneyness", "option_type"]

MASK_PROTOCOL_CONFIG = {
    "stress_test": {
        "base_hide_rate": 0.10,
        "node_weight": 1.00,
        "support_weight": 0.00,
    },
    "primary_realistic": {
        "base_hide_rate": 0.10,
        "node_weight": 0.65,
        "support_weight": 0.35,
    },
}

LOCKED_MASK_SEED = "nqfo-val-v1"
PHASE4_SHRINK_ALPHA = 0.25

OVERALL_TEST_MISSING_RATE = test["iv_observed"].isna().mean()

test_bucket_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(BUCKET_COLS)["is_missing"]
    .mean()
    .rename("test_bucket_missing_rate")
    .reset_index()
)

test_node_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(NODE_COLS)["is_missing"]
    .mean()
    .rename("test_node_missing_rate")
    .reset_index()
)

surface_levels = pd.concat(
    [
        train[["moneyness", "maturity_label", "maturity_days"]],
        test[["moneyness", "maturity_label", "maturity_days"]],
    ],
    ignore_index=True,
)

moneyness_levels = sorted(surface_levels["moneyness"].dropna().unique().tolist())
maturity_levels = (
    surface_levels[["maturity_label", "maturity_days"]]
    .drop_duplicates()
    .sort_values("maturity_days")["maturity_label"]
    .tolist()
)

m_idx = {m: i for i, m in enumerate(moneyness_levels)}
t_idx = {t: i for i, t in enumerate(maturity_levels)}


def stable_uniform(key: str) -> float:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def opposite_option(opt: str) -> str:
    return "put" if opt == "call" else "call"


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))


def local_support_profile(target_rows: pd.DataFrame, visible_rows: pd.DataFrame) -> pd.DataFrame:
    prof = target_rows.copy()

    visible_key_set = set(
        zip(
            visible_rows["date"],
            visible_rows["moneyness"],
            visible_rows["maturity_label"],
            visible_rows["option_type"],
        )
    )

    opp_visible = []
    same_maturity_adj_count = []
    same_moneyness_adj_count = []

    for d, m, t, o in zip(
        prof["date"],
        prof["moneyness"],
        prof["maturity_label"],
        prof["option_type"],
    ):
        opp_visible.append((d, m, t, opposite_option(o)) in visible_key_set)

        i = m_idx[m]
        j = t_idx[t]

        same_maturity_candidates = []
        if i - 1 >= 0:
            same_maturity_candidates.append((d, moneyness_levels[i - 1], t, o))
        if i + 1 < len(moneyness_levels):
            same_maturity_candidates.append((d, moneyness_levels[i + 1], t, o))

        same_moneyness_candidates = []
        if j - 1 >= 0:
            same_moneyness_candidates.append((d, m, maturity_levels[j - 1], o))
        if j + 1 < len(maturity_levels):
            same_moneyness_candidates.append((d, m, maturity_levels[j + 1], o))

        same_maturity_adj_count.append(sum(c in visible_key_set for c in same_maturity_candidates))
        same_moneyness_adj_count.append(sum(c in visible_key_set for c in same_moneyness_candidates))

    prof["opp_option_visible"] = opp_visible
    prof["same_maturity_adj_visible_count"] = same_maturity_adj_count
    prof["same_moneyness_adj_visible_count"] = same_moneyness_adj_count
    prof["local_support_score"] = (
        prof["opp_option_visible"].astype(int)
        + prof["same_maturity_adj_visible_count"]
        + prof["same_moneyness_adj_visible_count"]
    )
    return prof


def build_masked_validation_rows_with_protocol(
    full_df: pd.DataFrame,
    target_dates,
    protocol_name: str,
    seed: str = LOCKED_MASK_SEED,
) -> pd.DataFrame:
    cfg = MASK_PROTOCOL_CONFIG[protocol_name]

    val_df = full_df.loc[full_df["date"].isin(target_dates)].copy()
    val_df["is_orig_observed"] = val_df["iv_observed"].notna()
    val_df["is_orig_missing"] = ~val_df["is_orig_observed"]

    val_df = val_df.merge(test_bucket_pattern, on=BUCKET_COLS, how="left")
    val_df = val_df.merge(test_node_pattern, on=NODE_COLS, how="left")

    val_df["bucket_hide_rate_on_observed"] = (
        cfg["base_hide_rate"] * val_df["test_bucket_missing_rate"] / OVERALL_TEST_MISSING_RATE
    )

    val_df["priority_noise"] = val_df.apply(
        lambda r: stable_uniform(
            f"{seed}|{protocol_name}|{r['date'].date()}|{r['maturity_label']}|{r['moneyness']:.4f}|{r['option_type']}"
        ),
        axis=1,
    )

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    observed_support = local_support_profile(observed_pool, observed_pool)[["row_id", "local_support_score"]]
    val_df = val_df.merge(observed_support, on="row_id", how="left")
    val_df["local_support_score"] = val_df["local_support_score"].fillna(0).astype(int)

    val_df["is_pseudo_hidden"] = False

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    for _, g in observed_pool.groupby(["date", *BUCKET_COLS], sort=False):
        n_obs = len(g)
        n_hide = int(np.round(g["bucket_hide_rate_on_observed"].iloc[0] * n_obs))
        if n_hide <= 0:
            continue

        node_rank = g["test_node_missing_rate"].rank(method="average", pct=True)
        support_rank = g["local_support_score"].rank(method="average", pct=True)

        selection_priority = (
            cfg["node_weight"] * node_rank
            + cfg["support_weight"] * support_rank
            + 1e-6 * g["priority_noise"]
        )

        chosen_idx = (
            g.assign(selection_priority=selection_priority)
            .sort_values(["selection_priority", "row_id"], ascending=[False, True])
            .head(n_hide)
            .index
        )
        val_df.loc[chosen_idx, "is_pseudo_hidden"] = True

    val_df["is_scored_target"] = val_df["is_pseudo_hidden"]
    val_df["is_visible_anchor"] = val_df["is_orig_observed"] & ~val_df["is_pseudo_hidden"]
    val_df["is_effectively_missing"] = val_df["is_orig_missing"] | val_df["is_pseudo_hidden"]
    val_df["mask_protocol"] = protocol_name

    return val_df.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)


In [5]:
def build_node_lookup(observed_df: pd.DataFrame, pred_name: str) -> pd.DataFrame:
    return (
        observed_df.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename(pred_name)
        .reset_index()
    )


def predict_recent_node_median(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = target_rows.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    recent_dates = sorted(observed_train["date"].unique())[-lookback_dates:]
    recent_obs = observed_train.loc[observed_train["date"].isin(recent_dates)].copy()

    recent_lookup = build_node_lookup(recent_obs, "recent_node_pred")
    full_lookup = build_node_lookup(observed_train, "full_node_pred")

    out = out.merge(recent_lookup, on=NODE_COLS, how="left")
    out = out.merge(full_lookup, on=NODE_COLS, how="left")

    out["iv_pred"] = out["recent_node_pred"].fillna(out["full_node_pred"]).fillna(global_median)
    out["pred_source"] = np.select(
        [
            out["recent_node_pred"].notna(),
            out["full_node_pred"].notna(),
        ],
        [
            "recent_node_median",
            "full_node_median",
        ],
        default="global_median",
    )
    return out


def predict_same_date_linear_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, target_rows, lookback_dates=lookback_dates).copy()

    out["interp_pred"] = np.nan
    out["interp_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        x = anchors["moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        if len(anchors) == 1:
            interp_vals = np.repeat(y[0], len(g))
            interp_label = "same_date_single_anchor"
        else:
            interp_vals = np.interp(g["moneyness"].to_numpy(), x, y)
            interp_label = "same_date_linear_interp"

        out.loc[g.index, "interp_pred"] = interp_vals
        out.loc[g.index, "interp_source"] = interp_label

    use_interp = out["interp_pred"].notna()
    out.loc[use_interp, "iv_pred"] = out.loc[use_interp, "interp_pred"]
    out["pred_source"] = np.where(use_interp, out["interp_source"], out["pred_source"])

    return out


def predict_quadratic_smile_logm(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    degree: int = 2,
    min_anchors: int = 5,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["log_moneyness"] = np.log(out["moneyness"])
    out["smile_pred"] = np.nan
    out["smile_anchor_count"] = 0
    out["pred_source_smile"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["log_moneyness", "iv_observed"]]
            .dropna()
            .sort_values("log_moneyness")
        )

        if len(anchors) < min_anchors:
            continue
        if anchors["log_moneyness"].nunique() < degree + 1:
            continue

        x = anchors["log_moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        x_center = x.mean()
        coeffs = np.polyfit(x - x_center, y, deg=degree)

        target_x = g["log_moneyness"].to_numpy()
        pred = np.polyval(coeffs, target_x - x_center)

        in_range = (target_x >= x.min()) & (target_x <= x.max())
        pred = np.where(in_range, pred, np.nan)
        pred = np.clip(pred, pred_lo, pred_hi)

        out.loc[g.index, "smile_pred"] = pred
        out.loc[g.index, "smile_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_smile"] = np.where(in_range, "quadratic_smile_logm", pd.NA)

    use_smile = out["smile_pred"].notna()
    out.loc[use_smile, "iv_pred"] = out.loc[use_smile, "smile_pred"]
    out["pred_source"] = np.where(use_smile, out["pred_source_smile"], out["pred_source"])

    return out


def predict_total_variance_maturity_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["anchor_total_variance"] = np.where(
        out["iv_observed"].notna(),
        (out["iv_observed"] / 100.0) ** 2 * out["tau"],
        np.nan,
    )
    out["tv_pred"] = np.nan
    out["tv_anchor_count"] = 0
    out["pred_source_tv"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, moneyness, option_type), g_idx in out.groupby(
        ["date", "moneyness", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["tau", "anchor_total_variance"]]
            .dropna()
            .sort_values("tau")
        )

        if len(anchors) < 2:
            continue

        x = anchors["tau"].to_numpy()
        y = anchors["anchor_total_variance"].to_numpy()
        target_tau = g["tau"].to_numpy()

        in_range = (target_tau >= x.min()) & (target_tau <= x.max())
        pred_tv = np.interp(target_tau, x, y)
        pred_tv = np.where(in_range, pred_tv, np.nan)
        pred_tv = np.where(pred_tv > 0, pred_tv, np.nan)

        pred_iv = 100.0 * np.sqrt(pred_tv / target_tau)
        pred_iv = np.clip(pred_iv, pred_lo, pred_hi)

        out.loc[g.index, "tv_pred"] = pred_iv
        out.loc[g.index, "tv_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_tv"] = np.where(in_range, "tv_maturity_interp", pd.NA)

    use_tv = out["tv_pred"].notna()
    out.loc[use_tv, "iv_pred"] = out.loc[use_tv, "tv_pred"]
    out["pred_source"] = np.where(use_tv, out["pred_source_tv"], out["pred_source"])

    return out


def predict_structured_region_blend(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    smile_weight_center: float = 0.65,
    smile_weight_wing: float = 0.35,
) -> pd.DataFrame:
    base = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    smile = predict_quadratic_smile_logm(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "smile_pred"]].copy()

    tv = predict_total_variance_maturity_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "tv_pred"]].copy()

    out = base.merge(smile, on="row_id", how="left")
    out = out.merge(tv, on="row_id", how="left")

    out["wing_center_bucket"] = np.where(
        out["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )

    both_available = out["smile_pred"].notna() & out["tv_pred"].notna()
    only_smile = out["smile_pred"].notna() & out["tv_pred"].isna()
    only_tv = out["tv_pred"].notna() & out["smile_pred"].isna()

    center_mask = both_available & (out["wing_center_bucket"] == "center")
    wing_mask = both_available & (out["wing_center_bucket"] == "wing")

    out.loc[center_mask, "iv_pred"] = (
        smile_weight_center * out.loc[center_mask, "smile_pred"]
        + (1.0 - smile_weight_center) * out.loc[center_mask, "tv_pred"]
    )
    out.loc[wing_mask, "iv_pred"] = (
        smile_weight_wing * out.loc[wing_mask, "smile_pred"]
        + (1.0 - smile_weight_wing) * out.loc[wing_mask, "tv_pred"]
    )
    out.loc[only_smile, "iv_pred"] = out.loc[only_smile, "smile_pred"]
    out.loc[only_tv, "iv_pred"] = out.loc[only_tv, "tv_pred"]

    out["pred_source"] = np.select(
        [
            center_mask,
            wing_mask,
            only_smile,
            only_tv,
        ],
        [
            "structured_region_blend_center",
            "structured_region_blend_wing",
            "quadratic_smile_only",
            "tv_maturity_only",
        ],
        default=out["pred_source"],
    )

    return out


def predict_structured_region_blend_callput_shrink(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    shrink_alpha: float = PHASE4_SHRINK_ALPHA,
) -> pd.DataFrame:
    out = predict_structured_region_blend(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    opposite_visible = out.loc[
        out["is_visible_anchor"],
        ["date", "moneyness", "maturity_label", "option_type", "iv_observed"],
    ].copy()
    opposite_visible["option_type"] = opposite_visible["option_type"].map(opposite_option)
    opposite_visible = opposite_visible.rename(columns={"iv_observed": "opp_visible_iv"})

    out = out.merge(
        opposite_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    use_shrink = out["opp_visible_iv"].notna()
    out.loc[use_shrink, "iv_pred"] = (
        (1.0 - shrink_alpha) * out.loc[use_shrink, "iv_pred"]
        + shrink_alpha * out.loc[use_shrink, "opp_visible_iv"]
    )

    out["pred_source"] = np.where(
        use_shrink,
        "structured_region_blend_callput_shrink",
        out["pred_source"],
    )

    return out


In [6]:
ML_MIN_HISTORY_DATES = 20
ML_MASK_DATES_PER_FOLD = 20


def get_ml_candidate_dates_for_fold(
    fold_row,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
):
    fold_train_dates = train_date_summary.loc[:fold_row.train_end].index.to_list()
    eligible = fold_train_dates[min_history_dates:]
    return eligible[-n_mask_dates:]


def make_single_date_training_bundle(
    train_df: pd.DataFrame,
    target_date,
    protocol_name: str,
) -> dict:
    history_df = train_df.loc[train_df["date"] < target_date].copy()
    masked_target_date = build_masked_validation_rows_with_protocol(
        train_df,
        [target_date],
        protocol_name=protocol_name,
    )

    if history_df.empty:
        raise ValueError("History dataframe is empty. Choose a later target date.")

    if not (history_df["date"].max() < pd.Timestamp(target_date)):
        raise ValueError("Temporal leakage detected: history includes target date or later.")

    return {
        "target_date": pd.Timestamp(target_date),
        "protocol": protocol_name,
        "train_history": history_df,
        "masked_target_date": masked_target_date,
    }


def add_true_support_columns(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()

    if "local_support_score" in out.columns:
        out = out.rename(columns={"local_support_score": "mask_local_support_score"})

    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    support = local_support_profile(scored_targets, visible_anchors)[
        [
            "row_id",
            "opp_option_visible",
            "same_maturity_adj_visible_count",
            "same_moneyness_adj_visible_count",
            "local_support_score",
        ]
    ].copy()

    support = support.rename(columns={"local_support_score": "true_local_support_score"})
    support["any_local_same_date_support"] = (
        support["opp_option_visible"]
        | (support["same_maturity_adj_visible_count"] > 0)
        | (support["same_moneyness_adj_visible_count"] > 0)
    )
    support["hard_case"] = ~support["any_local_same_date_support"]

    out = out.merge(support, on="row_id", how="left")

    out["opp_option_visible"] = out["opp_option_visible"].astype("boolean").fillna(False).astype(bool)
    out["same_maturity_adj_visible_count"] = out["same_maturity_adj_visible_count"].fillna(0).astype(int)
    out["same_moneyness_adj_visible_count"] = out["same_moneyness_adj_visible_count"].fillna(0).astype(int)
    out["true_local_support_score"] = out["true_local_support_score"].fillna(0).astype(int)
    out["any_local_same_date_support"] = out["any_local_same_date_support"].astype("boolean").fillna(False).astype(bool)
    out["hard_case"] = out["hard_case"].astype("boolean").fillna(False).astype(bool)

    if "mask_local_support_score" in out.columns:
        out["mask_local_support_score"] = out["mask_local_support_score"].fillna(0).astype(int)

    return out


def add_same_maturity_anchor_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    out["left_anchor_iv"] = np.nan
    out["right_anchor_iv"] = np.nan
    out["left_anchor_dist"] = np.nan
    out["right_anchor_dist"] = np.nan
    out["same_maturity_visible_anchor_count"] = 0

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        anchor_x = anchors["moneyness"].to_numpy()
        anchor_y = anchors["iv_observed"].to_numpy()
        target_x = g["moneyness"].to_numpy()

        left_iv = []
        right_iv = []
        left_dist = []
        right_dist = []

        for x in target_x:
            left_mask = anchor_x <= x
            right_mask = anchor_x >= x

            if left_mask.any():
                lx = anchor_x[left_mask][-1]
                ly = anchor_y[left_mask][-1]
                left_iv.append(ly)
                left_dist.append(abs(x - lx))
            else:
                left_iv.append(np.nan)
                left_dist.append(np.nan)

            if right_mask.any():
                rx = anchor_x[right_mask][0]
                ry = anchor_y[right_mask][0]
                right_iv.append(ry)
                right_dist.append(abs(rx - x))
            else:
                right_iv.append(np.nan)
                right_dist.append(np.nan)

        out.loc[g.index, "left_anchor_iv"] = left_iv
        out.loc[g.index, "right_anchor_iv"] = right_iv
        out.loc[g.index, "left_anchor_dist"] = left_dist
        out.loc[g.index, "right_anchor_dist"] = right_dist
        out.loc[g.index, "same_maturity_visible_anchor_count"] = len(anchors)

    return out


def add_same_node_opposite_option_feature(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    opp_visible = (
        out.loc[out["is_visible_anchor"], ["date", "moneyness", "maturity_label", "option_type", "iv_observed"]]
        .copy()
    )
    opp_visible["option_type"] = opp_visible["option_type"].map(opposite_option)
    opp_visible = opp_visible.rename(columns={"iv_observed": "opp_visible_iv_same_node"})

    out = out.merge(
        opp_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    return out


def get_nearest_moneyness_rows(df: pd.DataFrame, target: float) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    distances = (df["moneyness"] - target).abs()
    min_dist = distances.groupby(df["date"]).transform("min")
    return df.loc[min_dist.eq(distances)].copy()


def add_date_level_regime_proxy_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()
    visible = out.loc[out["is_visible_anchor"] & out["iv_observed"].notna()].copy()

    total_rows = out.groupby("date").size().rename("date_total_row_count")
    visible_count = visible.groupby("date").size().rename("date_visible_anchor_count")
    visible_ratio = (visible_count / total_rows).rename("date_visible_anchor_ratio")
    visible_mean = visible.groupby("date")["iv_observed"].mean().rename("date_visible_iv_mean")
    visible_dispersion = visible.groupby("date")["iv_observed"].std().rename("date_visible_iv_dispersion")

    atm = (
        get_nearest_moneyness_rows(visible, 1.0)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_atm_iv_proxy")
    )
    left = (
        get_nearest_moneyness_rows(visible, 0.9)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_0p9_proxy")
    )
    right = (
        get_nearest_moneyness_rows(visible, 1.1)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_1p1_proxy")
    )

    maturity_means = (
        visible.groupby(["date", "maturity_days"])["iv_observed"]
        .mean()
        .reset_index()
        .sort_values(["date", "maturity_days"])
    )
    short_end = (
        maturity_means.groupby("date").first()["iv_observed"]
        .rename("date_short_end_iv_proxy")
    )
    long_end = (
        maturity_means.groupby("date").last()["iv_observed"]
        .rename("date_long_end_iv_proxy")
    )

    proxy_df = pd.concat(
        [
            total_rows,
            visible_count,
            visible_ratio,
            visible_mean,
            visible_dispersion,
            atm,
            left,
            right,
            short_end,
            long_end,
        ],
        axis=1,
    ).reset_index()

    proxy_df["date_visible_anchor_count"] = proxy_df["date_visible_anchor_count"].fillna(0).astype(int)
    proxy_df["date_visible_anchor_ratio"] = proxy_df["date_visible_anchor_ratio"].fillna(0.0)
    proxy_df["date_skew_proxy"] = proxy_df["date_iv_0p9_proxy"] - proxy_df["date_iv_1p1_proxy"]
    proxy_df["date_term_slope_proxy"] = (
        proxy_df["date_short_end_iv_proxy"] - proxy_df["date_long_end_iv_proxy"]
    )

    out = out.merge(proxy_df, on="date", how="left")
    return out


def build_feature_table_for_masked_bundle(
    train_history: pd.DataFrame,
    masked_target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    base = masked_target_rows.copy()

    linear_pred = predict_same_date_linear_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "same_date_linear_interp_pred"})

    smile_pred = predict_quadratic_smile_logm(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "smile_pred"]
    ].rename(columns={"smile_pred": "quadratic_smile_logm_pred"})

    tv_pred = predict_total_variance_maturity_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "tv_pred"]
    ].rename(columns={"tv_pred": "tv_maturity_interp_pred"})

    region_pred = predict_structured_region_blend(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred", "pred_source"]
    ].rename(
        columns={
            "iv_pred": "structured_region_blend_pred",
            "pred_source": "structured_region_blend_source",
        }
    )

    winner_pred = predict_structured_region_blend_callput_shrink(
        train_history,
        base,
        lookback_dates=lookback_dates,
        shrink_alpha=PHASE4_SHRINK_ALPHA,
    )[["row_id", "iv_pred", "pred_source"]].rename(
        columns={
            "iv_pred": "structured_winner_pred",
            "pred_source": "structured_winner_source",
        }
    )

    node_pred = predict_recent_node_median(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "recent_node_pred", "full_node_pred"]
    ]

    feat = base.merge(linear_pred, on="row_id", how="left")
    feat = feat.merge(smile_pred, on="row_id", how="left")
    feat = feat.merge(tv_pred, on="row_id", how="left")
    feat = feat.merge(region_pred, on="row_id", how="left")
    feat = feat.merge(winner_pred, on="row_id", how="left")
    feat = feat.merge(node_pred, on="row_id", how="left")

    feat = add_true_support_columns(feat)
    feat = add_same_maturity_anchor_features(feat)
    feat = add_same_node_opposite_option_feature(feat)
    feat = add_date_level_regime_proxy_features(feat)

    feat["support_score_gap"] = feat["true_local_support_score"] - feat.get("mask_local_support_score", 0)
    feat["has_opp_same_node_visible"] = feat["opp_visible_iv_same_node"].notna().astype(int)

    feat["log_moneyness"] = np.log(feat["moneyness"])
    feat["is_call"] = (feat["option_type"] == "call").astype(int)
    feat["wing_center_bucket"] = np.where(
        feat["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )
    feat["is_center"] = (feat["wing_center_bucket"] == "center").astype(int)
    feat["is_wing"] = (feat["wing_center_bucket"] == "wing").astype(int)
    feat["is_edge_maturity"] = feat["maturity_label"].isin(["1M", "6M"]).astype(int)
    feat["is_interior_maturity"] = 1 - feat["is_edge_maturity"]

    feat["smile_minus_linear"] = feat["quadratic_smile_logm_pred"] - feat["same_date_linear_interp_pred"]
    feat["tv_minus_linear"] = feat["tv_maturity_interp_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_linear"] = feat["structured_winner_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_region"] = feat["structured_winner_pred"] - feat["structured_region_blend_pred"]

    feat["target_iv"] = feat["iv_observed"]
    feat["target_residual"] = feat["target_iv"] - feat["structured_winner_pred"]

    return feat


def build_training_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
) -> pd.DataFrame:
    target_dates = get_ml_candidate_dates_for_fold(
        fold_row,
        min_history_dates=min_history_dates,
        n_mask_dates=n_mask_dates,
    )

    tables = []
    for target_date in target_dates:
        bundle = make_single_date_training_bundle(
            train_df,
            target_date=target_date,
            protocol_name=protocol_name,
        )

        feat = build_feature_table_for_masked_bundle(
            bundle["train_history"],
            bundle["masked_target_date"],
            lookback_dates=lookback_dates,
        )

        scored = feat.loc[feat["is_scored_target"]].copy()
        if not scored["is_orig_observed"].all():
            raise ValueError("Pseudo-masked training examples must come from originally observed rows only.")

        scored["target_date"] = pd.Timestamp(target_date)
        scored["protocol"] = protocol_name
        scored["fold"] = fold_row.fold
        scored["n_history_dates"] = bundle["train_history"]["date"].nunique()
        tables.append(scored)

    return pd.concat(tables, ignore_index=True)


def build_validation_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    val_dates = train_date_summary.loc[fold_row.val_start:fold_row.val_end].index.to_list()
    train_history = train_df.loc[train_df["date"] <= fold_row.train_end].copy()
    masked_val_rows = build_masked_validation_rows_with_protocol(
        train_df,
        val_dates,
        protocol_name=protocol_name,
    )

    feat = build_feature_table_for_masked_bundle(
        train_history,
        masked_val_rows,
        lookback_dates=lookback_dates,
    )

    scored = feat.loc[feat["is_scored_target"]].copy()
    if not scored["is_orig_observed"].all():
        raise ValueError("Validation scored rows must come from originally observed rows only.")

    scored["target_date"] = scored["date"]
    scored["protocol"] = protocol_name
    scored["fold"] = fold_row.fold
    scored["n_history_dates"] = train_history["date"].nunique()
    return scored.reset_index(drop=True)


def summarize_feature_table(feature_table: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n_rows": len(feature_table),
            "n_dates": feature_table["target_date"].nunique(),
            "date_min": feature_table["target_date"].min().date().isoformat(),
            "date_max": feature_table["target_date"].max().date().isoformat(),
            "target_iv_missing": int(feature_table["target_iv"].isna().sum()),
            "target_residual_missing": int(feature_table["target_residual"].isna().sum()),
            "winner_pred_missing": int(feature_table["structured_winner_pred"].isna().sum()),
            "mean_target_iv": feature_table["target_iv"].mean(),
            "mean_target_residual": feature_table["target_residual"].mean(),
            "rmse_structured_winner": rmse(feature_table["target_iv"], feature_table["structured_winner_pred"]),
        }
    )


def schema_alignment_report(train_table: pd.DataFrame, val_table: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    train_cols = set(train_table.columns)
    val_cols = set(val_table.columns)
    train_only = sorted(train_cols - val_cols)
    val_only = sorted(val_cols - train_cols)

    summary = pd.Series(
        {
            "n_train_cols": len(train_cols),
            "n_val_cols": len(val_cols),
            "n_common_cols": len(train_cols & val_cols),
            "n_train_only_cols": len(train_only),
            "n_val_only_cols": len(val_only),
            "schemas_match": len(train_only) == 0 and len(val_only) == 0,
        }
    )
    detail = pd.DataFrame(
        {
            "train_only_cols": pd.Series(train_only, dtype="object"),
            "val_only_cols": pd.Series(val_only, dtype="object"),
        }
    )
    return summary, detail


In [7]:
def rmse_from_series(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def prepare_design_matrices(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
):
    X_train = train_df.loc[:, feature_cols].copy()
    X_val = val_df.loc[:, feature_cols].copy()

    for df in (X_train, X_val):
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for col in bool_cols:
            df[col] = df[col].astype(int)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    if num_cols:
        train_num = X_train[num_cols].copy()
        val_num = X_val[num_cols].copy()

        medians = train_num.median(numeric_only=True)
        train_num = train_num.fillna(medians)
        val_num = val_num.fillna(medians)
    else:
        train_num = pd.DataFrame(index=X_train.index)
        val_num = pd.DataFrame(index=X_val.index)

    if cat_cols:
        train_cat = pd.get_dummies(X_train[cat_cols], dummy_na=True)
        val_cat = pd.get_dummies(X_val[cat_cols], dummy_na=True)
        train_cat, val_cat = train_cat.align(val_cat, join="outer", axis=1, fill_value=0)
    else:
        train_cat = pd.DataFrame(index=X_train.index)
        val_cat = pd.DataFrame(index=X_val.index)

    X_train_final = pd.concat([train_num, train_cat], axis=1)
    X_val_final = pd.concat([val_num, val_cat], axis=1)

    return X_train_final, X_val_final


def evaluate_ridge_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    alpha: float = 1.0,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "alpha": alpha,
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[["row_id", "target_date", "target_iv", "structured_winner_pred"]].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model, X_train.columns.tolist()


def evaluate_histgb_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    learning_rate: float = 0.05,
    max_depth: int = 3,
    max_iter: int = 300,
    min_samples_leaf: int = 10,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = HistGradientBoostingRegressor(
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_iter=max_iter,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "max_iter": max_iter,
        "min_samples_leaf": min_samples_leaf,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "target_residual",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model


In [8]:
FEATURE_GROUPS = {
    "row_local_core": [
        "moneyness",
        "log_moneyness",
        "maturity_days",
        "tau",
        "is_call",
        "is_center",
        "is_wing",
        "is_edge_maturity",
        "is_interior_maturity",
        "option_type",
        "maturity_label",
        "wing_center_bucket",
    ],
    "structured_predictions": [
        "structured_winner_pred",
        "structured_region_blend_pred",
        "tv_maturity_interp_pred",
        "quadratic_smile_logm_pred",
        "same_date_linear_interp_pred",
        "structured_winner_source",
        "structured_region_blend_source",
    ],
    "structured_gaps": [
        "smile_minus_linear",
        "tv_minus_linear",
        "winner_minus_linear",
        "winner_minus_region",
    ],
    "same_date_anchor_values": [
        "opp_visible_iv_same_node",
        "has_opp_same_node_visible",
        "left_anchor_iv",
        "right_anchor_iv",
        "left_anchor_dist",
        "right_anchor_dist",
        "same_maturity_visible_anchor_count",
    ],
    "support_geometry": [
        "opp_option_visible",
        "same_maturity_adj_visible_count",
        "same_moneyness_adj_visible_count",
        "true_local_support_score",
        "mask_local_support_score",
        "support_score_gap",
        "hard_case",
    ],
    "historical_priors": [
        "recent_node_pred",
        "full_node_pred",
    ],
    "date_regime_proxies": [
        "date_atm_iv_proxy",
        "date_skew_proxy",
        "date_term_slope_proxy",
        "date_visible_anchor_ratio",
        "date_visible_iv_dispersion",
        "date_visible_iv_mean",
        "date_visible_anchor_count",
    ],
}

FULL_HISTGB_ABLATION_FEATURES = sorted(
    {
        col
        for cols in FEATURE_GROUPS.values()
        for col in cols
    }
)


def feature_cols_from_groups(
    included_groups=None,
    excluded_groups=None,
) -> list[str]:
    included_groups = included_groups or list(FEATURE_GROUPS.keys())
    excluded_groups = excluded_groups or []

    included_cols = {
        col
        for group_name in included_groups
        for col in FEATURE_GROUPS[group_name]
    }
    excluded_cols = {
        col
        for group_name in excluded_groups
        for col in FEATURE_GROUPS[group_name]
    }

    final_cols = [col for col in FULL_HISTGB_ABLATION_FEATURES if col in included_cols and col not in excluded_cols]
    return final_cols


PHASE6_PRUNED_HISTGB_FEATURES = feature_cols_from_groups(
    included_groups=list(FEATURE_GROUPS.keys()),
    excluded_groups=["same_date_anchor_values", "date_regime_proxies"],
)

phase6_feature_registry = pd.DataFrame(
    [
        {
            "feature_set": "FULL_HISTGB_ABLATION_FEATURES",
            "n_raw_feature_cols": len(FULL_HISTGB_ABLATION_FEATURES),
            "notes": "Full standalone HistGB feature set from 05_1.",
        },
        {
            "feature_set": "PHASE6_PRUNED_HISTGB_FEATURES",
            "n_raw_feature_cols": len(PHASE6_PRUNED_HISTGB_FEATURES),
            "notes": "Pruned standalone nonlinear feature set: no same_date_anchor_values, no date_regime_proxies.",
        },
    ]
)

print("Phase 6 feature registry")
display(phase6_feature_registry)


Phase 6 feature registry


,feature_set,n_raw_feature_cols,notes
0,FULL_HISTGB_ABLATION_FEATURES,46,Full standalone HistGB feature set from 05_1.
1,PHASE6_PRUNED_HISTGB_FEATURES,32,Pruned standalone nonlinear feature set: no sa...


In [9]:
fold1_primary_train_table = build_training_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

fold1_primary_val_table = build_validation_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

print("Fold 1 / primary_realistic training table summary")
display(summarize_feature_table(fold1_primary_train_table).to_frame("value"))

print("Fold 1 / primary_realistic validation table summary")
display(summarize_feature_table(fold1_primary_val_table).to_frame("value"))

schema_summary, schema_detail = schema_alignment_report(
    fold1_primary_train_table,
    fold1_primary_val_table,
)

print("Train / validation schema alignment")
display(schema_summary.to_frame("value"))
display(schema_detail)

missing_pruned_cols = [
    col for col in PHASE6_PRUNED_HISTGB_FEATURES
    if col not in fold1_primary_train_table.columns
]
print("Missing pruned feature columns:", missing_pruned_cols)


Fold 1 / primary_realistic training table summary


,value
n_rows,155
n_dates,20
date_min,2025-03-24
date_max,2025-04-18
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,21.3784
mean_target_residual,0.0926
rmse_structured_winner,0.9005


Fold 1 / primary_realistic validation table summary


,value
n_rows,39
n_dates,5
date_min,2025-04-21
date_max,2025-04-25
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,19.5990
mean_target_residual,0.3840
rmse_structured_winner,1.3592


Train / validation schema alignment


,value
n_train_cols,74
n_val_cols,74
n_common_cols,74
n_train_only_cols,0
n_val_only_cols,0
schemas_match,True


,train_only_cols,val_only_cols


Missing pruned feature columns: []


In [10]:
structured_rmse_fold1_primary = rmse_from_series(
    fold1_primary_val_table["target_iv"],
    fold1_primary_val_table["structured_winner_pred"],
)

ridge_fold1_primary_summary, ridge_fold1_primary_preds, ridge_fold1_primary_model, ridge_fold1_primary_cols = evaluate_ridge_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_HISTGB_ABLATION_FEATURES,
    target_col="target_iv",
    alpha=1.0,
    prediction_mode="direct_iv",
)

pruned_histgb_fold1_primary_summary, pruned_histgb_fold1_primary_preds, pruned_histgb_fold1_primary_model = evaluate_histgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=PHASE6_PRUNED_HISTGB_FEATURES,
    target_col="target_residual",
    learning_rate=0.05,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    prediction_mode="residual",
)

phase6_fold1_primary_compare = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": structured_rmse_fold1_primary,
        },
        {
            "model": "ridge_direct_full",
            "val_rmse": ridge_fold1_primary_summary["val_rmse"],
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "val_rmse": pruned_histgb_fold1_primary_summary["val_rmse"],
        },
    ]
).sort_values("val_rmse")

print("Phase 6 Fold 1 / primary_realistic standalone finalist check")
display(phase6_fold1_primary_compare)


Phase 6 Fold 1 / primary_realistic standalone finalist check


,model,val_rmse
2,histgb_pruned_v1_no_anchor_no_regime,0.8428
1,ridge_direct_full,1.0556
0,structured_winner,1.3592


In [11]:
PROTOCOLS = ["primary_realistic", "stress_test"]

table_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = build_training_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        val_table = build_validation_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        table_cache[(protocol_name, fold_id)] = {
            "train": train_table,
            "val": val_table,
        }

print("Phase 6 cached tables")
display(
    pd.DataFrame(
        [
            {
                "protocol": protocol_name,
                "fold": fold_id,
                "n_train_rows": len(payload["train"]),
                "n_train_dates": payload["train"]["target_date"].nunique(),
                "n_val_rows": len(payload["val"]),
                "n_val_dates": payload["val"]["target_date"].nunique(),
                "schemas_match": set(payload["train"].columns) == set(payload["val"].columns),
            }
            for (protocol_name, fold_id), payload in table_cache.items()
        ]
    ).sort_values(["protocol", "fold"]).reset_index(drop=True)
)


Phase 6 cached tables


,protocol,fold,n_train_rows,n_train_dates,n_val_rows,n_val_dates,schemas_match
0,primary_realistic,1,155,20,39,5,True
1,primary_realistic,2,155,20,38,5,True
2,primary_realistic,3,155,20,39,5,True
3,primary_realistic,4,156,20,40,5,True
4,stress_test,1,155,20,39,5,True
5,stress_test,2,155,20,38,5,True
6,stress_test,3,155,20,39,5,True
7,stress_test,4,156,20,40,5,True


In [12]:
def run_structured_winner_baseline(train_table: pd.DataFrame, val_table: pd.DataFrame):
    pred_df = val_table[
        [
            "row_id",
            "target_date",
            "target_iv",
            "structured_winner_pred",
            "structured_winner_source",
            "maturity_label",
            "maturity_days",
            "option_type",
            "moneyness",
            "wing_center_bucket",
            "hard_case",
        ]
    ].copy()
    pred_df["iv_pred"] = pred_df["structured_winner_pred"]

    summary = {
        "model": "structured_winner",
        "target_col": "target_iv",
        "prediction_mode": "direct_iv",
        "feature_set": "(none)",
        "n_features_after_encoding": 0,
        "val_rmse": rmse_from_series(val_table["target_iv"], pred_df["iv_pred"]),
    }
    return summary, pred_df


def run_ridge_direct_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model, cols = evaluate_ridge_model(
        train_table,
        val_table,
        feature_cols=FULL_HISTGB_ABLATION_FEATURES,
        target_col="target_iv",
        alpha=1.0,
        prediction_mode="direct_iv",
    )
    pred_df = pred_df.copy()
    pred_df["feature_set"] = "FULL_HISTGB_ABLATION_FEATURES"

    summary = {
        "model": "ridge_direct_full",
        "feature_set": "FULL_HISTGB_ABLATION_FEATURES",
        **summary,
    }
    return summary, pred_df


def run_histgb_pruned_v1(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model = evaluate_histgb_model(
        train_table,
        val_table,
        feature_cols=PHASE6_PRUNED_HISTGB_FEATURES,
        target_col="target_residual",
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        prediction_mode="residual",
    )
    pred_df = pred_df.copy()
    pred_df["feature_set"] = "PHASE6_PRUNED_HISTGB_FEATURES"

    summary = {
        "model": "histgb_pruned_v1_no_anchor_no_regime",
        "feature_set": "PHASE6_PRUNED_HISTGB_FEATURES",
        **summary,
    }
    return summary, pred_df


PHASE6_STANDALONE_RUNNERS = {
    "structured_winner": run_structured_winner_baseline,
    "ridge_direct_full": run_ridge_direct_full,
    "histgb_pruned_v1_no_anchor_no_regime": run_histgb_pruned_v1,
}


In [14]:
phase6_standalone_rows = []
phase6_standalone_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        for model_name, runner in PHASE6_STANDALONE_RUNNERS.items():
            summary, pred_df = runner(train_table, val_table)

            phase6_standalone_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "n_train_rows": len(train_table),
                    "n_val_rows": len(val_table),
                    "structured_winner_rmse": rmse_from_series(
                        val_table["target_iv"],
                        val_table["structured_winner_pred"],
                    ),
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            phase6_standalone_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

phase6_standalone_results = (
    pd.DataFrame(phase6_standalone_rows)
    .sort_values(["protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("Phase 6 standalone finalist results")
display(phase6_standalone_results)


Phase 6 standalone finalist results


,protocol,fold,model,n_train_rows,n_val_rows,structured_winner_rmse,target_col,prediction_mode,feature_set,n_features_after_encoding,val_rmse,alpha,train_target_mean,learning_rate,max_depth,max_iter,min_samples_leaf
0,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,155,39,1.3592,target_residual,residual,PHASE6_PRUNED_HISTGB_FEATURES,51,0.8428,NaN,0.0926,0.0500,3.0000,300.0000,10.0000
1,primary_realistic,1,ridge_direct_full,155,39,1.3592,target_iv,direct_iv,FULL_HISTGB_ABLATION_FEATURES,65,1.0556,1.0000,21.3784,NaN,NaN,NaN,NaN
2,primary_realistic,1,structured_winner,155,39,1.3592,target_iv,direct_iv,(none),0,1.3592,NaN,NaN,NaN,NaN,NaN,NaN
3,primary_realistic,2,ridge_direct_full,155,38,1.0373,target_iv,direct_iv,FULL_HISTGB_ABLATION_FEATURES,65,0.9294,1.0000,20.7935,NaN,NaN,NaN,NaN
4,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,155,38,1.0373,target_residual,residual,PHASE6_PRUNED_HISTGB_FEATURES,51,0.9512,NaN,0.1957,0.0500,3.0000,300.0000,10.0000
5,primary_realistic,2,structured_winner,155,38,1.0373,target_iv,direct_iv,(none),0,1.0373,NaN,NaN,NaN,NaN,NaN,NaN
6,primary_realistic,3,structured_winner,155,39,0.7840,target_iv,direct_iv,(none),0,0.7840,NaN,NaN,NaN,NaN,NaN,NaN
7,primary_realistic,3,histgb_pruned_v1_no_anchor_no_regime,155,39,0.7840,target_residual,residual,PHASE6_PRUNED_HISTGB_FEATURES,51,0.8130,NaN,0.1308,0.0500,3.0000,300.0000,10.0000
8,primary_realistic,3,ridge_direct_full,155,39,0.7840,target_iv,direct_iv,FULL_HISTGB_ABLATION_FEATURES,65,0.8733,1.0000,21.9630,NaN,NaN,NaN,NaN
9,primary_realistic,4,ridge_direct_full,156,40,1.0748,target_iv,direct_iv,FULL_HISTGB_ABLATION_FEATURES,65,0.8397,1.0000,21.4115,NaN,NaN,NaN,NaN


In [15]:
phase6_standalone_protocol_summary = (
    phase6_standalone_results
    .groupby(["protocol", "model", "feature_set"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Phase 6 standalone protocol summary")
display(phase6_standalone_protocol_summary)


Phase 6 standalone protocol summary


,protocol,model,feature_set,mean_rmse,min_rmse,max_rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,PHASE6_PRUNED_HISTGB_FEATURES,0.8725,0.8130,0.9512
1,primary_realistic,ridge_direct_full,FULL_HISTGB_ABLATION_FEATURES,0.9245,0.8397,1.0556
2,primary_realistic,structured_winner,(none),1.0638,0.7840,1.3592
4,stress_test,ridge_direct_full,FULL_HISTGB_ABLATION_FEATURES,0.8463,0.7246,1.0449
3,stress_test,histgb_pruned_v1_no_anchor_no_regime,PHASE6_PRUNED_HISTGB_FEATURES,0.8989,0.8522,0.9839
5,stress_test,structured_winner,(none),1.1847,1.0614,1.3163


In [16]:
phase6_standalone_pivot = (
    phase6_standalone_protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

phase6_standalone_pivot["avg_two_protocols"] = (
    phase6_standalone_pivot["primary_realistic"] + phase6_standalone_pivot["stress_test"]
) / 2.0

phase6_standalone_pivot["max_two_protocols"] = phase6_standalone_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Phase 6 standalone decision pivot")
display(phase6_standalone_pivot.sort_values(["max_two_protocols", "avg_two_protocols"]))


Phase 6 standalone decision pivot


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8989,0.8857,0.8989
1,ridge_direct_full,0.9245,0.8463,0.8854,0.9245
2,structured_winner,1.0638,1.1847,1.1243,1.1847


In [17]:
structured_ref = (
    phase6_standalone_protocol_summary.loc[
        phase6_standalone_protocol_summary["model"] == "structured_winner",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

phase6_standalone_gain_table = (
    phase6_standalone_protocol_summary
    .merge(structured_ref, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Phase 6 standalone gain/loss vs structured winner")
display(phase6_standalone_gain_table)


Phase 6 standalone gain/loss vs structured winner


,protocol,model,feature_set,mean_rmse,min_rmse,max_rmse,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,PHASE6_PRUNED_HISTGB_FEATURES,0.8725,0.8130,0.9512,1.0638,-0.1913
1,primary_realistic,ridge_direct_full,FULL_HISTGB_ABLATION_FEATURES,0.9245,0.8397,1.0556,1.0638,-0.1393
2,primary_realistic,structured_winner,(none),1.0638,0.7840,1.3592,1.0638,0.0000
3,stress_test,ridge_direct_full,FULL_HISTGB_ABLATION_FEATURES,0.8463,0.7246,1.0449,1.1847,-0.3384
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,PHASE6_PRUNED_HISTGB_FEATURES,0.8989,0.8522,0.9839,1.1847,-0.2858
5,stress_test,structured_winner,(none),1.1847,1.0614,1.3163,1.1847,0.0000


In [18]:
phase6_standalone_fold_rank_view = (
    phase6_standalone_results[
        ["protocol", "fold", "model", "val_rmse"]
    ]
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Phase 6 standalone fold ranking")
display(phase6_standalone_fold_rank_view)


Phase 6 standalone fold ranking


,protocol,fold,model,val_rmse
0,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,0.8428
1,primary_realistic,1,ridge_direct_full,1.0556
2,primary_realistic,1,structured_winner,1.3592
3,primary_realistic,2,ridge_direct_full,0.9294
4,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,0.9512
5,primary_realistic,2,structured_winner,1.0373
6,primary_realistic,3,structured_winner,0.7840
7,primary_realistic,3,histgb_pruned_v1_no_anchor_no_regime,0.8130
8,primary_realistic,3,ridge_direct_full,0.8733
9,primary_realistic,4,ridge_direct_full,0.8397


In [19]:
phase6_standalone_registry = (
    phase6_candidate_registry.loc[
        phase6_candidate_registry["model"].isin(
            [
                "structured_winner",
                "ridge_direct_full",
                "histgb_pruned_v1_no_anchor_no_regime",
            ]
        )
    ]
    .merge(phase6_standalone_pivot, on="model", how="left")
    .sort_values(["max_two_protocols", "avg_two_protocols"])
    .reset_index(drop=True)
)

print("Phase 6 standalone registry")
display(phase6_standalone_registry)


Phase 6 standalone registry


,model,family,target_mode,routing_style,nonlinear_branch,changed_from_phase5,status_entering_phase6,notes,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,histgb_pruned_v1_no_anchor_no_regime,tree_ml,residual,single_model,pruned_histgb_residual,Replaces full HistGB residual as the main stan...,standalone_reference,Best standalone nonlinear reference entering P...,0.8725,0.8989,0.8857,0.8989
1,ridge_direct_full,linear_ml,direct_iv,single_model,(none),No change.,standalone_reference,Best linear standalone reference and conservat...,0.9245,0.8463,0.8854,0.9245
2,structured_winner,structured,direct_iv,single_model,(none),No change.,benchmark,Locked structured benchmark from Phase 4/5.,1.0638,1.1847,1.1243,1.1847


In [20]:
def run_histgb_residual_full_reference(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model = evaluate_histgb_model(
        train_table,
        val_table,
        feature_cols=FULL_HISTGB_ABLATION_FEATURES,
        target_col="target_residual",
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        prediction_mode="residual",
    )
    pred_df = pred_df.copy()
    pred_df["feature_set"] = "FULL_HISTGB_ABLATION_FEATURES"

    summary = {
        "model": "histgb_residual_full",
        "feature_set": "FULL_HISTGB_ABLATION_FEATURES",
        **summary,
    }
    return summary, pred_df


In [21]:
phase6_backward_reference_rows = []
phase6_backward_reference_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        summary, pred_df = run_histgb_residual_full_reference(train_table, val_table)

        phase6_backward_reference_rows.append(
            {
                "protocol": protocol_name,
                "fold": fold_id,
                "model": "histgb_residual_full",
                **summary,
            }
        )

        pred_df = pred_df.copy()
        pred_df["protocol"] = protocol_name
        pred_df["fold"] = fold_id
        pred_df["model"] = "histgb_residual_full"
        phase6_backward_reference_cache[(protocol_name, fold_id, "histgb_residual_full")] = pred_df

phase6_backward_reference_results = (
    pd.DataFrame(phase6_backward_reference_rows)
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Phase 6 backward-reference nonlinear branch results")
display(phase6_backward_reference_results)


Phase 6 backward-reference nonlinear branch results


,protocol,fold,model,feature_set,target_col,prediction_mode,learning_rate,max_depth,max_iter,min_samples_leaf,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,primary_realistic,1,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.0926,0.8903,1.3592
1,primary_realistic,2,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.1957,0.8908,1.0373
2,primary_realistic,3,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.1308,0.8861,0.7840
3,primary_realistic,4,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.2026,0.8171,1.0748
4,stress_test,1,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.3156,1.1223,1.3163
5,stress_test,2,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,65,0.2944,0.8688,1.1719
6,stress_test,3,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,64,0.1984,0.9210,1.0614
7,stress_test,4,histgb_residual_full,FULL_HISTGB_ABLATION_FEATURES,target_residual,residual,0.0500,3,300,10,64,0.1883,0.8574,1.1892


In [22]:
def build_prediction_panel_for_fold_protocol(
    protocol_name: str,
    fold_id: int,
    table_cache: dict,
    standalone_prediction_cache: dict,
    backward_reference_cache: dict,
) -> pd.DataFrame:
    val_table = table_cache[(protocol_name, fold_id)]["val"].copy()

    base_cols = [
        "row_id",
        "target_date",
        "target_iv",
        "option_type",
        "maturity_label",
        "maturity_days",
        "moneyness",
        "wing_center_bucket",
        "hard_case",
        "structured_winner_source",
    ]

    panel = val_table[base_cols].copy()

    structured_pred = standalone_prediction_cache[(protocol_name, fold_id, "structured_winner")][
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "structured_winner__iv_pred"})

    ridge_pred = standalone_prediction_cache[(protocol_name, fold_id, "ridge_direct_full")][
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "ridge_direct_full__iv_pred"})

    pruned_histgb_pred = standalone_prediction_cache[(protocol_name, fold_id, "histgb_pruned_v1_no_anchor_no_regime")][
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "histgb_pruned_v1_no_anchor_no_regime__iv_pred"})

    full_histgb_pred = backward_reference_cache[(protocol_name, fold_id, "histgb_residual_full")][
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "histgb_residual_full__iv_pred"})

    panel = panel.merge(structured_pred, on="row_id", how="left")
    panel = panel.merge(ridge_pred, on="row_id", how="left")
    panel = panel.merge(pruned_histgb_pred, on="row_id", how="left")
    panel = panel.merge(full_histgb_pred, on="row_id", how="left")

    return panel


In [23]:
def apply_hybrid_center_structured_else_histgb_pruned(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_center_structured_else_histgb_pruned"
    out["hybrid_route"] = "histgb_pruned_v1_no_anchor_no_regime"
    out["iv_pred"] = out["histgb_pruned_v1_no_anchor_no_regime__iv_pred"]

    structured_mask = out["wing_center_bucket"] == "center"
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_slice_no_hard_case_override_pruned(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_hard_case_override_pruned"
    out["hybrid_route"] = "histgb_pruned_v1_no_anchor_no_regime"
    out["iv_pred"] = out["histgb_pruned_v1_no_anchor_no_regime__iv_pred"]

    ridge_mask = out["structured_winner_source"] == "tv_maturity_only"
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_center_structured_else_histgb(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_center_structured_else_histgb"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    structured_mask = out["wing_center_bucket"] == "center"
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_slice_no_hard_case_override(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_hard_case_override"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    ridge_mask = out["structured_winner_source"] == "tv_maturity_only"
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


PHASE6_HYBRID_BUILDERS = {
    "hybrid_center_structured_else_histgb_pruned": apply_hybrid_center_structured_else_histgb_pruned,
    "hybrid_slice_no_hard_case_override_pruned": apply_hybrid_slice_no_hard_case_override_pruned,
    "hybrid_center_structured_else_histgb": apply_hybrid_center_structured_else_histgb,
    "hybrid_slice_no_hard_case_override": apply_hybrid_slice_no_hard_case_override,
}


In [24]:
phase6_hybrid_rows = []
phase6_hybrid_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        panel_df = build_prediction_panel_for_fold_protocol(
            protocol_name,
            fold_id,
            table_cache=table_cache,
            standalone_prediction_cache=phase6_standalone_prediction_cache,
            backward_reference_cache=phase6_backward_reference_cache,
        )

        for model_name, builder in PHASE6_HYBRID_BUILDERS.items():
            pred_df = builder(panel_df).copy()

            val_rmse = rmse_from_series(pred_df["target_iv"], pred_df["iv_pred"])
            val_mae = float((pred_df["target_iv"] - pred_df["iv_pred"]).abs().mean())

            phase6_hybrid_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "n_val_rows": len(pred_df),
                    "val_rmse": val_rmse,
                    "val_mae": val_mae,
                }
            )

            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            phase6_hybrid_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

phase6_hybrid_results = (
    pd.DataFrame(phase6_hybrid_rows)
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Phase 6 hybrid finalist results")
display(phase6_hybrid_results)


Phase 6 hybrid finalist results


,protocol,fold,model,n_val_rows,val_rmse,val_mae
0,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,39,0.7600,0.6206
1,primary_realistic,1,hybrid_slice_no_hard_case_override,39,0.8072,0.6223
2,primary_realistic,1,hybrid_center_structured_else_histgb_pruned,39,0.8334,0.6698
3,primary_realistic,1,hybrid_center_structured_else_histgb,39,0.8921,0.6930
4,primary_realistic,2,hybrid_center_structured_else_histgb,38,0.8595,0.7045
5,primary_realistic,2,hybrid_slice_no_hard_case_override,38,0.8599,0.6949
6,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,38,0.9063,0.7201
7,primary_realistic,2,hybrid_center_structured_else_histgb_pruned,38,0.9198,0.7266
8,primary_realistic,3,hybrid_center_structured_else_histgb_pruned,39,0.7622,0.6628
9,primary_realistic,3,hybrid_slice_no_hard_case_override_pruned,39,0.7866,0.6765


In [25]:
FINAL_PHASE6_MODELS = [
    "structured_winner",
    "ridge_direct_full",
    "histgb_pruned_v1_no_anchor_no_regime",
    "hybrid_center_structured_else_histgb_pruned",
    "hybrid_slice_no_hard_case_override_pruned",
    "hybrid_center_structured_else_histgb",
    "hybrid_slice_no_hard_case_override",
]

phase6_finalist_rows = pd.concat(
    [
        phase6_standalone_results[["protocol", "fold", "model", "val_rmse"]],
        phase6_hybrid_results[["protocol", "fold", "model", "val_rmse"]],
    ],
    ignore_index=True,
)

phase6_finalist_rows = phase6_finalist_rows.loc[
    phase6_finalist_rows["model"].isin(FINAL_PHASE6_MODELS)
].copy()

phase6_final_protocol_summary = (
    phase6_finalist_rows
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Phase 6 finalist protocol summary")
display(phase6_final_protocol_summary)


Phase 6 finalist protocol summary


,protocol,model,mean_rmse,min_rmse,max_rmse
4,primary_realistic,hybrid_slice_no_hard_case_override_pruned,0.8158,0.7600,0.9063
3,primary_realistic,hybrid_slice_no_hard_case_override,0.8323,0.8072,0.8599
2,primary_realistic,hybrid_center_structured_else_histgb_pruned,0.8341,0.7622,0.9198
1,primary_realistic,hybrid_center_structured_else_histgb,0.8575,0.8221,0.8921
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8130,0.9512
5,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556
6,primary_realistic,structured_winner,1.0638,0.7840,1.3592
11,stress_test,hybrid_slice_no_hard_case_override_pruned,0.7917,0.7127,0.8407
10,stress_test,hybrid_slice_no_hard_case_override,0.8054,0.7316,0.8645
12,stress_test,ridge_direct_full,0.8463,0.7246,1.0449


In [26]:
phase6_final_pivot = (
    phase6_final_protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

phase6_final_pivot["avg_two_protocols"] = (
    phase6_final_pivot["primary_realistic"] + phase6_final_pivot["stress_test"]
) / 2.0

phase6_final_pivot["max_two_protocols"] = phase6_final_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Phase 6 finalist decision pivot")
display(phase6_final_pivot.sort_values(["max_two_protocols", "avg_two_protocols"]))


Phase 6 finalist decision pivot


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
4,hybrid_slice_no_hard_case_override_pruned,0.8158,0.7917,0.8037,0.8158
3,hybrid_slice_no_hard_case_override,0.8323,0.8054,0.8189,0.8323
2,hybrid_center_structured_else_histgb_pruned,0.8341,0.8750,0.8545,0.8750
1,hybrid_center_structured_else_histgb,0.8575,0.8967,0.8771,0.8967
0,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8989,0.8857,0.8989
5,ridge_direct_full,0.9245,0.8463,0.8854,0.9245
6,structured_winner,1.0638,1.1847,1.1243,1.1847


In [27]:
phase6_backward_compare = phase6_final_pivot.loc[
    phase6_final_pivot["model"].isin(
        [
            "hybrid_center_structured_else_histgb_pruned",
            "hybrid_slice_no_hard_case_override_pruned",
            "hybrid_center_structured_else_histgb",
            "hybrid_slice_no_hard_case_override",
        ]
    )
].copy()

print("Phase 6 backward hybrid comparison")
display(phase6_backward_compare.sort_values(["max_two_protocols", "avg_two_protocols"]))


Phase 6 backward hybrid comparison


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
4,hybrid_slice_no_hard_case_override_pruned,0.8158,0.7917,0.8037,0.8158
3,hybrid_slice_no_hard_case_override,0.8323,0.8054,0.8189,0.8323
2,hybrid_center_structured_else_histgb_pruned,0.8341,0.8750,0.8545,0.8750
1,hybrid_center_structured_else_histgb,0.8575,0.8967,0.8771,0.8967


In [28]:
phase6_final_registry = (
    phase6_candidate_registry
    .merge(phase6_final_pivot, on="model", how="left")
    .sort_values(["max_two_protocols", "avg_two_protocols"])
    .reset_index(drop=True)
)

print("Phase 6 finalist registry")
display(phase6_final_registry)


Phase 6 finalist registry


,model,family,target_mode,routing_style,nonlinear_branch,changed_from_phase5,status_entering_phase6,notes,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,hybrid_slice_no_hard_case_override_pruned,hybrid,mixed,slice_gated,histgb_pruned_v1_no_anchor_no_regime,Updated from full HistGB branch to pruned Hist...,provisional_hybrid,Phase 6 confirmation candidate; default pruned...,0.8158,0.7917,0.8037,0.8158
1,hybrid_slice_no_hard_case_override,hybrid,mixed,slice_gated,histgb_residual_full,No change; retained only for brief backward co...,backward_reference,Old provisional hybrid winner from 05_0; needs...,0.8323,0.8054,0.8189,0.8323
2,hybrid_center_structured_else_histgb_pruned,hybrid,mixed,center_vs_wing,histgb_pruned_v1_no_anchor_no_regime,Updated from full HistGB branch to pruned Hist...,provisional_hybrid,Phase 6 confirmation candidate; structured in ...,0.8341,0.8750,0.8545,0.8750
3,hybrid_center_structured_else_histgb,hybrid,mixed,center_vs_wing,histgb_residual_full,No change; retained only for brief backward co...,backward_reference,Old full-HistGB-based hybrid for one-time comp...,0.8575,0.8967,0.8771,0.8967
4,histgb_pruned_v1_no_anchor_no_regime,tree_ml,residual,single_model,pruned_histgb_residual,Replaces full HistGB residual as the main stan...,standalone_reference,Best standalone nonlinear reference entering P...,0.8725,0.8989,0.8857,0.8989
5,ridge_direct_full,linear_ml,direct_iv,single_model,(none),No change.,standalone_reference,Best linear standalone reference and conservat...,0.9245,0.8463,0.8854,0.9245
6,structured_winner,structured,direct_iv,single_model,(none),No change.,benchmark,Locked structured benchmark from Phase 4/5.,1.0638,1.1847,1.1243,1.1847


In [29]:
PHASE6_TOP3_FINALISTS = [
    "hybrid_slice_no_hard_case_override_pruned",
    "histgb_pruned_v1_no_anchor_no_regime",
    "ridge_direct_full",
]

phase6_diag_rows = []

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])
        val_table = table_cache[(protocol_name, fold_id)]["val"].copy()

        base = val_table[
            [
                "row_id",
                "target_date",
                "target_iv",
                "maturity_label",
                "maturity_days",
                "option_type",
                "moneyness",
                "wing_center_bucket",
                "hard_case",
                "structured_winner_source",
                "structured_winner_pred",
            ]
        ].copy()

        for model_name in PHASE6_TOP3_FINALISTS:
            if model_name == "hybrid_slice_no_hard_case_override_pruned":
                pred_df = phase6_hybrid_prediction_cache[(protocol_name, fold_id, model_name)].copy()
                pred_cols = ["row_id", "iv_pred", "hybrid_route"]
            else:
                pred_df = phase6_standalone_prediction_cache[(protocol_name, fold_id, model_name)].copy()
                pred_cols = ["row_id", "iv_pred"]

            merged = base.merge(
                pred_df[pred_cols],
                on="row_id",
                how="left",
            )

            merged["protocol"] = protocol_name
            merged["fold"] = fold_id
            merged["model"] = model_name
            merged["abs_error"] = (merged["target_iv"] - merged["iv_pred"]).abs()
            merged["sq_error"] = (merged["target_iv"] - merged["iv_pred"]) ** 2

            if "hybrid_route" not in merged.columns:
                merged["hybrid_route"] = pd.NA

            phase6_diag_rows.append(merged)

phase6_diag_df = pd.concat(phase6_diag_rows, ignore_index=True)

print("Phase 6 finalist diagnostics dataframe")
display(phase6_diag_df.head())


Phase 6 finalist diagnostics dataframe


,row_id,target_date,target_iv,maturity_label,maturity_days,option_type,moneyness,wing_center_bucket,hard_case,structured_winner_source,structured_winner_pred,iv_pred,hybrid_route,protocol,fold,model,abs_error,sq_error
0,9262,2025-04-21,19.2304,1M,30,call,1.1000,center,False,structured_region_blend_callput_shrink,19.1951,19.1951,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.0353,0.0012
1,9245,2025-04-21,24.2840,1M,30,put,0.8750,wing,False,quadratic_smile_only,25.1451,25.1451,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.8611,0.7415
2,9294,2025-04-21,19.0918,2M,60,call,1.1250,wing,False,structured_region_blend_callput_shrink,18.4505,18.2708,histgb_pruned_v1_no_anchor_no_regime,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.8210,0.6740
3,9281,2025-04-21,20.2648,2M,60,put,0.9500,center,False,structured_region_blend_callput_shrink,20.5633,20.5633,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.2985,0.0891
4,9312,2025-04-21,18.7062,3M,91,call,0.9750,center,False,structured_region_blend_callput_shrink,19.3098,19.3098,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.6036,0.3643


In [30]:
def slice_error_table_local(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)

    out = (
        df.groupby(["protocol", "model", *group_cols], dropna=False)
        .agg(
            n=("row_id", "size"),
            mae=("abs_error", "mean"),
            mean_sq_error=("sq_error", "mean"),
        )
        .reset_index()
    )

    out["rmse"] = np.sqrt(out["mean_sq_error"])
    return out.drop(columns=["mean_sq_error"]).sort_values(["protocol", "model", *group_cols]).reset_index(drop=True)


In [31]:
phase6_top3_wing_center = slice_error_table_local(
    phase6_diag_df,
    group_cols=["wing_center_bucket"],
)

print("Phase 6 top-3 finalists | wing vs center")
display(phase6_top3_wing_center)


Phase 6 top-3 finalists | wing vs center


,protocol,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,center,65,0.6043,0.7645
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,wing,91,0.7871,0.9439
2,primary_realistic,hybrid_slice_no_hard_case_override_pruned,center,65,0.5069,0.6537
3,primary_realistic,hybrid_slice_no_hard_case_override_pruned,wing,91,0.7657,0.9159
4,primary_realistic,ridge_direct_full,center,65,0.6021,0.7847
5,primary_realistic,ridge_direct_full,wing,91,0.7887,1.0174
6,stress_test,histgb_pruned_v1_no_anchor_no_regime,center,49,0.5953,0.7387
7,stress_test,histgb_pruned_v1_no_anchor_no_regime,wing,107,0.7643,0.9651
8,stress_test,hybrid_slice_no_hard_case_override_pruned,center,49,0.5279,0.6479
9,stress_test,hybrid_slice_no_hard_case_override_pruned,wing,107,0.6967,0.8520


In [32]:
phase6_top3_maturity = slice_error_table_local(
    phase6_diag_df,
    group_cols=["maturity_label"],
)

print("Phase 6 top-3 finalists | maturity")
display(phase6_top3_maturity)


Phase 6 top-3 finalists | maturity


,protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.7523,0.9516
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.7090,0.8688
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,3M,39,0.7484,0.8758
3,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,6M,38,0.6312,0.7862
4,primary_realistic,hybrid_slice_no_hard_case_override_pruned,1M,40,0.6375,0.8477
5,primary_realistic,hybrid_slice_no_hard_case_override_pruned,2M,39,0.7112,0.8714
6,primary_realistic,hybrid_slice_no_hard_case_override_pruned,3M,39,0.6906,0.7924
7,primary_realistic,hybrid_slice_no_hard_case_override_pruned,6M,38,0.5911,0.7483
8,primary_realistic,ridge_direct_full,1M,40,0.8074,1.0851
9,primary_realistic,ridge_direct_full,2M,39,0.6356,0.9238


In [33]:
phase6_hybrid_route_mix = (
    phase6_diag_df.loc[
        phase6_diag_df["model"] == "hybrid_slice_no_hard_case_override_pruned"
    ]
    .groupby(["protocol", "hybrid_route"], as_index=False)["row_id"]
    .count()
    .rename(columns={"row_id": "count"})
    .sort_values(["protocol", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Phase 6 leading hybrid | route mix")
display(phase6_hybrid_route_mix)


Phase 6 leading hybrid | route mix


,protocol,hybrid_route,count
0,primary_realistic,structured_winner,88
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,58
2,primary_realistic,ridge_direct_full,10
3,stress_test,structured_winner,74
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,69
5,stress_test,ridge_direct_full,13


In [34]:
phase6_top3_wing_center_pivot = (
    phase6_top3_wing_center
    .pivot_table(
        index=["protocol", "wing_center_bucket"],
        columns="model",
        values="rmse",
    )
    .reset_index()
)

print("Phase 6 top-3 finalists | wing vs center pivot")
display(phase6_top3_wing_center_pivot)


Phase 6 top-3 finalists | wing vs center pivot


model,protocol,wing_center_bucket,histgb_pruned_v1_no_anchor_no_regime,hybrid_slice_no_hard_case_override_pruned,ridge_direct_full
0,primary_realistic,center,0.7645,0.6537,0.7847
1,primary_realistic,wing,0.9439,0.9159,1.0174
2,stress_test,center,0.7387,0.6479,0.7958
3,stress_test,wing,0.9651,0.8520,0.8797


In [38]:
phase6_top3_maturity_pivot = (
    phase6_top3_maturity
    .pivot_table(
        index=["protocol", "maturity_label"],
        columns="model",
        values="rmse",
    )
    .reset_index()
)

print("Phase 6 top-3 finalists | maturity pivot")
display(phase6_top3_maturity_pivot)


Phase 6 top-3 finalists | maturity pivot


model,protocol,maturity_label,histgb_pruned_v1_no_anchor_no_regime,hybrid_slice_no_hard_case_override_pruned,ridge_direct_full
0,primary_realistic,1M,0.9516,0.8477,1.0851
1,primary_realistic,2M,0.8688,0.8714,0.9238
2,primary_realistic,3M,0.8758,0.7924,0.8905
3,primary_realistic,6M,0.7862,0.7483,0.7767
4,stress_test,1M,0.7883,0.7185,0.8091
5,stress_test,2M,1.0110,0.8956,0.9070
6,stress_test,3M,0.7919,0.6599,0.7974
7,stress_test,6M,0.9898,0.8786,0.8998


In [39]:
def add_delta_vs_leader(pivot_df: pd.DataFrame, leader_col: str) -> pd.DataFrame:
    out = pivot_df.copy()

    for col in out.columns:
        if col in ["protocol", "wing_center_bucket", "maturity_label", leader_col]:
            continue
        out[f"{col}_minus_{leader_col}"] = out[col] - out[leader_col]

    return out


In [40]:
phase6_top3_wing_center_delta = add_delta_vs_leader(
    phase6_top3_wing_center_pivot,
    "hybrid_slice_no_hard_case_override_pruned",
)

print("Phase 6 top-3 finalists | wing vs center delta vs leader")
display(phase6_top3_wing_center_delta)


Phase 6 top-3 finalists | wing vs center delta vs leader


model,protocol,wing_center_bucket,histgb_pruned_v1_no_anchor_no_regime,hybrid_slice_no_hard_case_override_pruned,ridge_direct_full,histgb_pruned_v1_no_anchor_no_regime_minus_hybrid_slice_no_hard_case_override_pruned,ridge_direct_full_minus_hybrid_slice_no_hard_case_override_pruned
0,primary_realistic,center,0.7645,0.6537,0.7847,0.1108,0.1310
1,primary_realistic,wing,0.9439,0.9159,1.0174,0.0279,0.1015
2,stress_test,center,0.7387,0.6479,0.7958,0.0908,0.1479
3,stress_test,wing,0.9651,0.8520,0.8797,0.1130,0.0277


In [41]:
phase6_top3_maturity_delta = add_delta_vs_leader(
    phase6_top3_maturity_pivot,
    "hybrid_slice_no_hard_case_override_pruned",
)

print("Phase 6 top-3 finalists | maturity delta vs leader")
display(phase6_top3_maturity_delta)


Phase 6 top-3 finalists | maturity delta vs leader


model,protocol,maturity_label,histgb_pruned_v1_no_anchor_no_regime,hybrid_slice_no_hard_case_override_pruned,ridge_direct_full,histgb_pruned_v1_no_anchor_no_regime_minus_hybrid_slice_no_hard_case_override_pruned,ridge_direct_full_minus_hybrid_slice_no_hard_case_override_pruned
0,primary_realistic,1M,0.9516,0.8477,1.0851,0.1039,0.2374
1,primary_realistic,2M,0.8688,0.8714,0.9238,-0.0026,0.0524
2,primary_realistic,3M,0.8758,0.7924,0.8905,0.0834,0.0981
3,primary_realistic,6M,0.7862,0.7483,0.7767,0.0379,0.0284
4,stress_test,1M,0.7883,0.7185,0.8091,0.0697,0.0906
5,stress_test,2M,1.0110,0.8956,0.9070,0.1153,0.0113
6,stress_test,3M,0.7919,0.6599,0.7974,0.1320,0.1375
7,stress_test,6M,0.9898,0.8786,0.8998,0.1113,0.0213


## Phase 6.0 final candidate confirmation: summary

This notebook was the confirmation step for the finalist model set entering Phase 6.

The key reason this notebook was needed is that Phase 5.1 changed the best standalone nonlinear reference from:

- `histgb_residual_full`

to:

- `histgb_pruned_v1_no_anchor_no_regime`

So the main question here was:

**Does the hybrid still win after rebuilding it with the pruned HistGB branch?**

---

## 1. What this notebook confirmed

This notebook rebuilt the finalist candidates in a fresh, self-contained setup using the same leakage-safe train / validation pipeline established in Phase 5.

The confirmed finalist set evaluated here was:

- `structured_winner`
- `ridge_direct_full`
- `histgb_pruned_v1_no_anchor_no_regime`
- `hybrid_center_structured_else_histgb_pruned`
- `hybrid_slice_no_hard_case_override_pruned`

For backward reference only, this notebook also compared the earlier full-HistGB-based hybrids:

- `hybrid_center_structured_else_histgb`
- `hybrid_slice_no_hard_case_override`

---

## 2. Main result

The updated hybrid:

- `hybrid_slice_no_hard_case_override_pruned`

is the confirmed leader from this notebook.

Its protocol-level performance is:

- `primary_realistic = 0.8158`
- `stress_test = 0.7917`
- `avg_two_protocols = 0.8037`
- `max_two_protocols = 0.8158`

This makes it the strongest candidate in the confirmed Phase 6 field.

---

## 3. Why this result matters

The most important confirmation is:

### The hybrid still wins after replacing the old nonlinear branch with the pruned HistGB branch

This means the earlier hybrid advantage was not just an artifact of the older full HistGB standalone model.

In fact, the branch update improved the hybrid.

Backward comparison:

- old `hybrid_slice_no_hard_case_override`
  - `primary_realistic = 0.8323`
  - `stress_test = 0.8054`

- updated `hybrid_slice_no_hard_case_override_pruned`
  - `primary_realistic = 0.8158`
  - `stress_test = 0.7917`

So the updated hybrid is better on both protocols.

---

## 4. Confirmed model landscape after 06.0

### Confirmed benchmark
- `structured_winner`

### Confirmed standalone linear reference
- `ridge_direct_full`

### Confirmed standalone nonlinear reference
- `histgb_pruned_v1_no_anchor_no_regime`

### Confirmed leading hybrid
- `hybrid_slice_no_hard_case_override_pruned`

### Other hybrids
- `hybrid_center_structured_else_histgb_pruned` remains informative but is not the leading candidate
- the earlier full-HistGB-based hybrids are now backward references, not active finalists

---

## 5. Final ranking from this notebook

The confirmed ordering is:

1. `hybrid_slice_no_hard_case_override_pruned`
2. `hybrid_slice_no_hard_case_override`
3. `hybrid_center_structured_else_histgb_pruned`
4. `hybrid_center_structured_else_histgb`
5. `histgb_pruned_v1_no_anchor_no_regime`
6. `ridge_direct_full`
7. `structured_winner`

The important practical interpretation is:

- the best hybrid is clearly ahead of the best standalone nonlinear reference
- the best standalone nonlinear reference is still valuable as a simpler challenger
- Ridge remains the conservative linear specialist
- the structured winner remains the benchmark

---

## 6. Decision going into 06.1

The models advancing to final training-policy selection and narrow tuning are:

- `hybrid_slice_no_hard_case_override_pruned`
- `histgb_pruned_v1_no_anchor_no_regime`
- `ridge_direct_full`

The structured winner remains in the next notebook as an untuned benchmark.

The non-advancing hybrids are retained only as references.

---

## 7. Main conclusion from 06.0

This notebook resolves the main Phase 6 confirmation question.

The answer is:

**Yes, the hybrid still wins after rebuilding it with the pruned HistGB nonlinear branch.**

So the updated pruned-branch hybrid becomes the main finalist entering Phase 6.1:

- `hybrid_slice_no_hard_case_override_pruned`

The next step is therefore:

- choose the final shared ML training policy
- run narrow finalist-only tuning
- and determine whether this confirmed hybrid remains the final locked inference model


## 6A. Compact finalist slice diagnostics

A compact finalist diagnostic block was also run for:

- `hybrid_slice_no_hard_case_override_pruned`
- `histgb_pruned_v1_no_anchor_no_regime`
- `ridge_direct_full`

This showed:

- the leading hybrid beats both standalone challengers in both:
  - center
  - wing
  under both protocols
- the leading hybrid beats both standalone challengers on all `stress_test` maturity buckets
- under `primary_realistic`, the only small giveback is a very small loss to the pruned HistGB model on `2M`
- the route mix is coherent and non-degenerate:
  - structured branch remains heavily used
  - pruned HistGB branch is meaningfully used
  - Ridge override remains narrow and targeted

So the `06_0` winner is not being driven by a single narrow slice. The advantage is broad enough to justify advancing it as the main finalist.
